# Python 与 LangGraph 交互式学习笔记

这份 Notebook 是 `python_langgraph_notes.md` 的配套版本：Markdown 单元负责解释概念，Code 单元负责演示可运行示例。

使用建议：

- 先顺序阅读 Markdown，再运行后面的代码单元。
- 大部分示例不需要真实大模型 API Key。
- DeepSeek 相关单元会显式读取项目根目录 `.env`，支持 `DEEPSEEK_API_KEY`，也兼容 OpenAI-style 的 `OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`。
- 当前 LangGraph v1 之后推荐使用 `from langchain.agents import create_agent`，不要再用已弃用的 `langgraph.prebuilt.create_react_agent`。


## 1. LangGraph 是什么

LangGraph 是 LangChain 生态中用于构建有状态 Agent 和复杂 LLM 工作流的框架。

简单理解：

```text
LangChain 更偏组件：模型、Prompt、工具、链、检索器。
LangGraph 更偏编排：状态、节点、边、循环、暂停、恢复、多代理。
```

当你的应用需要工具调用、多轮状态、人工审批、流式过程、多 Agent 协作、恢复和调试时，LangGraph 就比单纯的链式调用更合适。


In [7]:
# 安装依赖示例。Notebook 中一般不自动执行安装命令。
# %pip install -U langgraph langchain langchain-deepseek langchain-openai python-dotenv

import os
from pathlib import Path

from dotenv import load_dotenv

# Notebook 的工作目录可能是 docs/，也可能是项目根目录。
# 这里向上查找 .env，确保能读到 D:/PythonProject/LearnOne/.env。
def find_project_env(start: Path | None = None) -> Path | None:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            return env_path
    return None


env_path = find_project_env()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"已加载环境变量：{env_path}")
else:
    print("没有找到 .env，真实模型调用单元会跳过。")

print("DEEPSEEK_API_KEY 已配置：", bool(os.getenv("DEEPSEEK_API_KEY")))
print("OPENAI_API_KEY 已配置：", bool(os.getenv("OPENAI_API_KEY")))
print("LangGraph 学习 Notebook 已加载。")


已加载环境变量：D:\PythonProject\LearnOne\.env
DEEPSEEK_API_KEY 已配置： True
OPENAI_API_KEY 已配置： True
LangGraph 学习 Notebook 已加载。


## 2.1 DeepSeek 连接自检

如果真实模型调用失败，先运行这个单元。它会用当前 `.env` 做一次最小 DeepSeek 请求，确认是配置问题、网络问题，还是接口临时 502。


In [8]:
import os
import time

from langchain_deepseek import ChatDeepSeek

# 这里强制使用 DeepSeek 官方分支，不读取 OPENAI_BASE_URL，避免误走 /anthropic 兼容地址。
model_name = os.getenv("DEEPSEEK_MODEL") or "deepseek-chat"
timeout = float(os.getenv("DEEPSEEK_TIMEOUT", "30"))

print("DEEPSEEK_API_KEY 已配置：", bool(os.getenv("DEEPSEEK_API_KEY")))
print("使用模型：", model_name)
print("DEEPSEEK_API_BASE：", os.getenv("DEEPSEEK_API_BASE") or "默认 DeepSeek 地址")
print("忽略 OPENAI_BASE_URL：", os.getenv("OPENAI_BASE_URL"))

try:
    start = time.time()
    check_model = ChatDeepSeek(
        model=model_name,
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_API_BASE"),
        temperature=0,
        timeout=timeout,
        max_retries=0,
    )
    result = check_model.invoke("只回答两个字：成功")
    print("连接成功，用时：", round(time.time() - start, 2), "秒")
    print("返回内容：", result.content)
except Exception as exc:
    print("连接失败。")
    print("错误类型：", type(exc).__name__)
    print("错误信息：", exc)
    print("如果是 502，多数是服务端网关临时错误，稍后重试即可。")


DEEPSEEK_API_KEY 已配置： True
使用模型： deepseek-chat
DEEPSEEK_API_BASE： 默认 DeepSeek 地址
忽略 OPENAI_BASE_URL： https://api.deepseek.com/anthropic
连接成功，用时： 0.72 秒
返回内容： 成功


## 2. DeepSeek + create_agent 基础写法

`create_agent` 是当前推荐的预构建 Agent 入口。这个单元只演示真实 DeepSeek 的普通对话调用，不绑定工具。

原因：真实模型的 tool calling 受模型、网关和接口兼容性影响，可能出现 502 等服务端错误。Notebook 里的工具调用和 context 会在后面的离线假模型单元演示，保证学习过程稳定。


In [9]:
import os

from langchain.agents import create_agent


def build_deepseek_model():
    """优先使用 langchain-deepseek，兼容 OpenAI-style DeepSeek 配置。"""

    # Notebook 里真实请求容易受网络影响，所以显式配置超时时间和重试次数。
    request_timeout = float(os.getenv("DEEPSEEK_TIMEOUT", "60"))
    max_retries = int(os.getenv("DEEPSEEK_MAX_RETRIES", "2"))

    # 方式一：优先读取 DeepSeek 官方环境变量。
    # 目录结构和 .env 已经配置好时，一般只需要保证 DEEPSEEK_API_KEY 存在。
    if os.getenv("DEEPSEEK_API_KEY"):
        from langchain_deepseek import ChatDeepSeek

        return ChatDeepSeek(
            # 模型名默认 deepseek-chat，也可以在 .env 里用 DEEPSEEK_MODEL 覆盖。
            model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            # 温度越低，输出越稳定，适合教学和工具调用。
            temperature=float(os.getenv("DEEPSEEK_TEMPERATURE", "0")),
            # timeout 控制单次请求最长等待时间，避免网络慢时一直卡住。
            timeout=request_timeout,
            # max_retries 控制超时/网络波动时自动重试次数。
            max_retries=max_retries,
        )

    # 方式二：兼容 OpenAI-style 配置，适合把 DeepSeek 当 OpenAI 兼容接口使用。
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=os.getenv("OPENAI_MODEL", "deepseek-chat"),
            api_key=os.getenv("OPENAI_API_KEY"),
            # DeepSeek 兼容接口地址，可在 .env 中通过 OPENAI_BASE_URL 覆盖。
            base_url=os.getenv("OPENAI_BASE_URL", "https://api.deepseek.com/v1"),
            temperature=0,
            timeout=request_timeout,
            max_retries=max_retries,
        )

    # 没有 key 时返回 None，后续单元用 if model 判断，避免 Notebook 直接报错。
    return None


model = build_deepseek_model()

if model is None:
    print("没有找到 DeepSeek/OpenAI-compatible API Key，跳过真实模型调用。")
else:
    # create_agent 会创建预构建 Agent。tools=[] 表示这里只演示普通模型回答。
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个简洁可靠的中文助手。",
    )

    try:
        # Agent 输入通常是 messages 字段，最后一条消息一般是最终回答。
        response = agent.invoke(
            {"messages": [{"role": "user", "content": "用一句话解释 LangGraph 是什么。"}]}
        )
        print(response["messages"][-1].content)
    except Exception as exc:
        # 超时通常是网络、代理、接口繁忙或 base_url 配置导致，不是 LangGraph 语法错误。
        print("真实模型请求失败，后续离线示例仍可继续学习。")
        print("错误类型：", type(exc).__name__)
        print("错误信息：", exc)
        print("可尝试：检查网络/代理/base_url，或在 .env 中调大 DEEPSEEK_TIMEOUT=120。")


LangGraph 是一个用于构建有状态、多步骤语言模型工作流的框架，支持通过图结构定义和编排复杂的 AI 代理逻辑。


## 3. Agent 输入输出

LangGraph / LangChain Agent 的常见输入是：

```python
{"messages": [{"role": "user", "content": "你好"}]}
```

输出通常也是一个字典，其中最常用的是 `messages`。如果配置了结构化输出，还会有 `structured_response`。


## 4. Context 上下文：把运行时信息安全传给工具

`context` 适合传用户 ID、租户、权限、语言偏好、请求来源等运行时信息。

重点：这些信息不是让模型自己编，而是由代码在 `agent.invoke(..., context=...)` 时传入，工具通过 `ToolRuntime[Context]` 读取。

下面这个例子不需要真实模型。它用一个离线假模型主动发起工具调用，从而完整演示 context 如何进入工具。真实 DeepSeek 工具调用可以参考项目里的 `langGraph/TestOne.py`，但 Notebook 里用离线模型避免接口 502 影响学习。


In [2]:
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.runnables import Runnable
from langchain_core.tools import BaseTool


@dataclass
class Context:
    user_id: str


USER_PROFILES = {
    "U001": "张三，上海销售",
    "U002": "李四，北京财务",
}


@tool
def get_user_profile(runtime: ToolRuntime[Context]) -> str:
    """读取当前登录用户资料。"""

    user_id = runtime.context.user_id
    profile = USER_PROFILES.get(user_id, "未知用户")
    return f"当前用户 {user_id}：{profile}。"


class ContextDemoModel(BaseChatModel):
    """离线演示用模型：第一次请求工具，第二次总结工具结果。"""

    calls: int = 0

    @property
    def _llm_type(self) -> str:
        return "context-demo-model"

    def bind_tools(
        self,
        tools: Sequence[dict[str, Any] | type | BaseTool | Any],
        *,
        tool_choice: str | None = None,
        **kwargs: Any,
    ) -> Runnable:
        return self

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any | None = None,
        **kwargs: Any,
    ) -> ChatResult:
        self.calls += 1
        if self.calls == 1:
            message = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "get_user_profile",
                        "args": {},
                        "id": "call-get-user-profile",
                        "type": "tool_call",
                    }
                ],
            )
        else:
            tool_message = next(
                (message for message in reversed(messages) if message.type == "tool"),
                None,
            )
            content = f"工具返回：{tool_message.content if tool_message else '无工具结果'}"
            message = AIMessage(content=content)
        return ChatResult(generations=[ChatGeneration(message=message)])


context_agent = create_agent(
    model=ContextDemoModel(),
    tools=[get_user_profile],
    context_schema=Context,
    system_prompt="你是一个演示 runtime context 的助手。",
)

response = context_agent.invoke(
    {"messages": [{"role": "user", "content": "读取当前用户资料"}]},
    context=Context(user_id="U001"),
)

print(response["messages"][-1].content)


工具返回：当前用户 U001：张三，上海销售。


上面的 context 传递链路是：

```text
Context(user_id="U001")
-> agent.invoke(..., context=context)
-> Agent 执行 get_user_profile 工具
-> ToolRuntime[Context]
-> runtime.context.user_id
-> "U001"
```

这就是 context 的核心价值：把运行时信息安全地交给工具，而不是让模型自己猜。


## 5. StateGraph：状态、节点、边

LangGraph 的底层核心是图：

- `State`：当前状态快照。
- `Node`：处理状态的 Python 函数。
- `Edge`：决定下一步去哪个节点。

下面是最小状态图。


In [4]:
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class SimpleState(TypedDict):
    # question 是调用 graph.invoke 时传入的输入字段。
    question: str
    # answer 是 answer_node 执行后写入的输出字段。
    answer: str


def answer_node(state: SimpleState) -> dict[str, str]:
    # 节点接收完整 state，返回“局部状态更新”。
    # 注意：节点要返回 dict，而不是直接返回字符串。
    return {"answer": f"你问的是：{state['question']}"}


# 创建图构建器，并声明状态结构。
builder = StateGraph(SimpleState)

# 注册节点：节点名 answer，对应函数 answer_node。
builder.add_node("answer", answer_node)

# START 是图入口，表示运行后第一步进入 answer。
builder.add_edge(START, "answer")

# answer 执行完进入 END，表示流程结束。
builder.add_edge("answer", END)

# compile 后才得到可运行的 graph。
simple_graph = builder.compile()

# 传入初始 state，图会从 START 开始执行。
simple_graph.invoke({"question": "什么是 LangGraph？"})


{'question': '什么是 LangGraph？', 'answer': '你问的是：什么是 LangGraph？'}

## 6. 条件边：根据状态决定下一步

固定边适合线性流程，条件边适合分支流程。下面的图会根据 `score` 判断走 `pass_node` 还是 `fail_node`。


In [6]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class ScoreState(TypedDict):
    # 输入分数。
    score: int
    # 路由后的结果。
    result: str


def route_by_score(state: ScoreState) -> Literal["pass_node", "fail_node"]:
    # 条件边函数只负责返回下一个节点名。
    return "pass_node" if state["score"] >= 60 else "fail_node"


def pass_node(state: ScoreState) -> dict[str, str]:
    # 通过分支写入 result。
    return {"result": "通过"}


def fail_node(state: ScoreState) -> dict[str, str]:
    # 未通过分支写入 result。
    return {"result": "未通过"}


builder = StateGraph(ScoreState)

# 从 START 出发时，不走固定边，而是调用 route_by_score 判断进入哪个节点。
builder.add_conditional_edges(START, route_by_score)

# 注册两个分支节点。
builder.add_node("pass_node", pass_node)
builder.add_node("fail_node", fail_node)

# 两个分支最终都结束。
builder.add_edge("pass_node", END)
builder.add_edge("fail_node", END)

score_graph = builder.compile()
print(score_graph.invoke({"score": 88, "result": ""}))
print(score_graph.invoke({"score": 42, "result": ""}))


{'score': 88, 'result': '通过'}
{'score': 42, 'result': '未通过'}


## 7. State reducer：让列表追加而不是覆盖

默认情况下，节点返回的新字段会覆盖旧值。对于消息列表、日志列表这类字段，通常希望追加。可以用 `Annotated[..., reducer]` 指定合并方式。


In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    # Annotated[list[str], add] 表示多个节点返回 logs 时使用 add 合并。
    # 对 list 来说，operator.add 等价于列表拼接，而不是覆盖。
    logs: Annotated[list[str], add]


def step_one(state: LogState) -> dict[str, list[str]]:
    # 返回新的日志片段，reducer 会追加到旧 logs 后面。
    return {"logs": ["执行 step_one"]}


def step_two(state: LogState) -> dict[str, list[str]]:
    return {"logs": ["执行 step_two"]}


builder = StateGraph(LogState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)

# 串行流程：START -> step_one -> step_two -> END。
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

log_graph = builder.compile()
log_graph.invoke({"logs": ["开始"]})


## 8. 持久化：checkpointer + thread_id

短期记忆依赖检查点。核心是：

- 创建图时传入 `checkpointer`。
- 调用图时传入 `config={"configurable": {"thread_id": "..."}}`。
- 相同 `thread_id` 会接上同一条线程。


In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class MemoryState(TypedDict):
    # events 使用 add reducer，新的事件会追加到旧事件列表。
    events: Annotated[list[str], add]


def remember_node(state: MemoryState) -> dict[str, list[str]]:
    # 每次执行节点都追加一条事件。
    return {"events": ["本轮执行了一次"]}


builder = StateGraph(MemoryState)
builder.add_node("remember", remember_node)
builder.add_edge(START, "remember")
builder.add_edge("remember", END)

# compile 时加入 checkpointer，图就能按 thread_id 保存状态。
memory_graph = builder.compile(checkpointer=InMemorySaver())

# 同一个 thread_id 表示同一个线程/会话。
config = {"configurable": {"thread_id": "demo-thread"}}

first = memory_graph.invoke({"events": ["开始"]}, config=config)
second = memory_graph.invoke({"events": ["继续"]}, config=config)

print("第一次：", first)
print("第二次：", second)


## 8.1 DeepSeek + MessagesState：最小聊天图

这个例子对应图片里的“线程隔离的持久化层”前置代码：先创建一个最小的 `MessagesState` 图，节点负责调用 DeepSeek，并把模型回复追加回 `messages`。

注意：这里使用你的 `DEEPSEEK_API_KEY` 和 `DEEPSEEK_API_BASE`。如果你的 `DEEPSEEK_API_BASE` 已经包含 `/v1`，就不要再额外拼接 `/v1`。


In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph

# 读取项目根目录或当前环境中的 .env 配置。
load_dotenv(override=True)

# Notebook 真实请求显式配置 timeout / max_retries，降低偶发超时影响。
request_timeout = float(os.getenv("DEEPSEEK_TIMEOUT", "60"))
max_retries = int(os.getenv("DEEPSEEK_MAX_RETRIES", "2"))

# 使用 DeepSeek 聊天模型。
model = ChatDeepSeek(
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
    timeout=request_timeout,
    max_retries=max_retries,
)


def call_model(state: MessagesState):
    # MessagesState 的核心字段是 messages。
    # 模型读取已有消息，返回一条 AIMessage。
    response = model.invoke(state["messages"])
    # 返回 messages 更新，LangGraph 会把新消息追加进状态。
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

chat_graph = builder.compile()

try:
    chat_graph.invoke({"messages": [{"role": "user", "content": "hi 你是谁？"}]})
except Exception as exc:
    print("真实 DeepSeek 请求失败，图结构已经创建成功。")
    print("错误类型：", type(exc).__name__)
    print("错误信息：", exc)
    print("可继续运行后面的离线图示例；如果要真实调用，请检查网络并调大 DEEPSEEK_TIMEOUT。")


## 8.2 线程隔离的持久化层

`checkpointer` 会保存图执行过程中的中间状态。调用时传入 `thread_id` 后，相同线程会接着之前的消息历史继续，不同线程互相隔离。

图片里写的是 `MemorySaver`，在当前版本里它就是 `InMemorySaver` 的别名。学习阶段可以用内存版；生产环境要换成数据库或 LangGraph Platform 的持久化能力。


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver 是内存检查点，适合 Notebook 演示。
memory = InMemorySaver()

# persistent_graph 和上面的 chat_graph 逻辑一样，只是多了持久化能力。
persistent_graph = builder.compile(checkpointer=memory)

# thread_id 决定这次运行属于哪条对话线程。
config = {"configurable": {"thread_id": "thread-1"}}
input_message = {"role": "user", "content": "hi! 我是 tomie"}

try:
    for chunk in persistent_graph.stream(
        {"messages": [input_message]},
        config=config,
        # values 表示每次返回完整状态快照，便于观察 messages 变化。
        stream_mode="values",
    ):
        chunk["messages"][-1].pretty_print()
except Exception as exc:
    print("真实模型请求失败，但持久化图已经创建成功。")
    print("错误类型：", type(exc).__name__)
    print("错误信息：", exc)


In [ ]:
# 同一个 thread_id 会保留上一轮上下文。
try:
    for chunk in persistent_graph.stream(
        {"messages": [{"role": "user", "content": "我刚才告诉你我叫什么？"}]},
        config=config,
        stream_mode="values",
    ):
        chunk["messages"][-1].pretty_print()
except Exception as exc:
    print("真实模型请求失败，通常是接口超时或网络问题。")
    print("错误类型：", type(exc).__name__)
    print("错误信息：", exc)


In [ ]:
# 换一个 thread_id，就是另一条独立线程。
other_config = {"configurable": {"thread_id": "thread-2"}}

for chunk in persistent_graph.stream(
    {"messages": [{"role": "user", "content": "我刚才告诉你我叫什么？"}]},
    config=other_config,
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()


## 8.3 跨线程共享长期记忆

`thread_id` 解决的是“同一条会话线程”的短期记忆。长期记忆通常要按 `user_id` 存储，这样不同线程也能共享用户资料。

下面先给一个离线可运行的 `InMemoryStore` 例子：不需要 embedding，不需要调用模型，只演示 `user_id -> memory` 的读写。


In [ ]:
from langgraph.store.memory import InMemoryStore

# InMemoryStore 用来演示长期记忆；数据只存在当前进程内。
store = InMemoryStore()

user_id = "user-tomie"

# namespace 用来隔离不同用户或不同记忆类型。
namespace = ("memories", user_id)

# 写入一条用户画像记忆。
store.put(
    namespace,
    "profile",
    {"text": "Tomie 喜欢 Python，也在学习 LangGraph。"},
)

# 写入一条偏好记忆。
store.put(
    namespace,
    "preference",
    {"text": "Tomie 希望示例尽量使用 DeepSeek。"},
)

# 读取该 namespace 下的所有记忆。
memories = store.search(namespace)
for item in memories:
    print(item.key, item.value)


如果你的网关支持 OpenAI 兼容的 embedding 接口，也可以给 `InMemoryStore` 配置索引。图片里类似使用 `OpenAIEmbeddings(model="Pro/BAAI/bge-m3")`，这里保留 DeepSeek/兼容网关写法。运行前请确认你的 `DEEPSEEK_API_BASE` 是否提供 embedding 模型。


In [ ]:
# 可选：带 embedding 索引的长期记忆。
# 如果你的 DEEPSEEK_API_BASE 不支持 embedding，请跳过这个单元。
import os
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

embedding_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(
            model="Pro/BAAI/bge-m3",
            api_key=os.environ.get("DEEPSEEK_API_KEY"),
            base_url=os.environ.get("DEEPSEEK_API_BASE"),
        ),
        "dims": 1024,
    }
)

embedding_store.put(
    ("memories", "user-tomie"),
    "langgraph",
    {"text": "Tomie 正在学习 LangGraph 的持久化和人机交互。"},
)

embedding_store.search(("memories", "user-tomie"), query="用户在学什么？")


## 8.4 短期记忆：最小 ReAct 图

这一组图展示的是一个最小 ReAct 智能体：

- `agent` 节点调用模型。
- 如果模型返回 `tool_calls`，条件边走到 `action`。
- `action` 使用 `ToolNode` 执行工具。
- 工具结果回到 `agent`，直到模型不再调用工具。

这里的 `search` 是一个模拟工具，不会真的联网。


In [ ]:
from typing import Literal

from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode


@tool
def search(query: str) -> str:
    """调用此函数可以浏览网络。"""
    # 这里是离线模拟结果，真实项目可以替换成搜索 API。
    return "北京天气晴朗，大约22度，湿度30%。"


# ToolNode 负责真正执行模型请求的工具调用。
tools = [search]
tool_node = ToolNode(tools)

# bind_tools 后，模型才知道可以调用哪些工具。
bound_model = model.bind_tools(tools)


def should_continue(state: MessagesState) -> Literal["action", "__end__"]:
    """根据最后一条 AI 消息是否包含 tool_calls 决定下一步。"""
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return END
    return "action"


def call_react_model(state: MessagesState):
    # 调用绑定工具后的模型，模型可能返回 tool_calls。
    response = bound_model.invoke(state["messages"])
    return {"messages": [response]}


workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_react_model)
workflow.add_node("action", tool_node)
workflow.add_edge(START, "agent")

# agent 后面走条件边：需要工具就去 action，否则结束。
workflow.add_conditional_edges("agent", should_continue, ["action", END])

# 工具执行完再回到 agent，让模型观察工具结果并继续回答。
workflow.add_edge("action", "agent")

react_graph = workflow.compile(checkpointer=InMemorySaver())


In [ ]:
react_config = {"configurable": {"thread_id": "react-thread-1"}}

try:
    for chunk in react_graph.stream(
        {"messages": [{"role": "user", "content": "北京天气怎么样？需要的话可以搜索。"}]},
        config=react_config,
        stream_mode="values",
    ):
        chunk["messages"][-1].pretty_print()
except Exception as exc:
    print("真实模型请求失败，ReAct 图结构仍可作为代码学习参考。")
    print("错误类型：", type(exc).__name__)
    print("错误信息：", exc)


## 8.5 优化记忆

长对话不能无限把所有历史消息塞给模型。常见做法有两类：

- 消息过滤：对旧消息做删除、裁剪或编辑，防止上下文爆炸。
- 消息总结：把旧消息压缩成摘要，保留关键信息。

记忆管理本质上是在“召回率”和“精度”之间做平衡：保留太少会忘事，保留太多会增加成本并干扰回答。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage

old_messages = [
    HumanMessage(content="我叫 Tomie。"),
    AIMessage(content="你好 Tomie。"),
    HumanMessage(content="我喜欢 Python。"),
    AIMessage(content="记住了，你喜欢 Python。"),
    HumanMessage(content="我正在学习 LangGraph。"),
]

# 过滤策略：删除较早消息，只保留最近两条。
remove_old_messages = [RemoveMessage(id=message.id) for message in old_messages[:-2]]
kept_messages = old_messages[-2:]

print("要删除的消息数量：", len(remove_old_messages))
print("保留的消息：")
for message in kept_messages:
    message.pretty_print()

# 总结策略：把历史压缩成一条系统消息。
summary_message = SystemMessage(
    content="对话摘要：用户叫 Tomie，喜欢 Python，正在学习 LangGraph。"
)
summary_message.pretty_print()


## 8.6 人机交互：审查工具调用

人机交互常见位置：

- 批准或拒绝：LLM 给出计划后，由人决定继续还是改走其他节点。
- 状态编辑：人工修改 state 或工具参数，再继续执行。
- 工具审查：工具真正执行前，人工确认工具名和参数是否正确。
- 多轮对话：Agent 和人之间反复交换信息。

下面是一个离线可运行的“工具审查”例子：模型节点先模拟产生一个工具调用，`human_review` 节点通过 `interrupt` 暂停，人工用 `Command(resume=...)` 选择继续、更新参数或反馈给模型。


In [ ]:
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt


@tool
def review_search(query: str) -> str:
    """模拟搜索工具。"""
    return f"搜索结果：{query} -> 北京天气晴朗，约22度，湿度30%。"


review_tools = [review_search]
review_tool_node = ToolNode(review_tools)


class ReviewState(MessagesState):
    """简单状态。"""


def call_llm_for_review(state: ReviewState):
    """为了离线稳定运行，这里模拟 LLM 生成一个工具调用。"""
    return {
        "messages": [
            AIMessage(
                content="我需要查询天气。",
                id="ai-search-request",
                tool_calls=[
                    {
                        "name": "review_search",
                        "args": {"query": "北京天气"},
                        "id": "call_search_1",
                        "type": "tool_call",
                    }
                ],
            )
        ]
    }


def human_review_node(
    state: ReviewState,
) -> Command[Literal["call_llm", "run_tool"]]:
    last_message = state["messages"][-1]
    tool_call = last_message.tool_calls[-1]

    human_review = interrupt(
        {
            "question": "这个工具调用正确吗？",
            "tool_call": tool_call,
        }
    )

    review_action = human_review["action"]
    review_data = human_review.get("data")

    if review_action == "continue":
        return Command(goto="run_tool")

    if review_action == "update":
        updated_message = AIMessage(
            content=last_message.content,
            id=last_message.id,
            tool_calls=[
                {
                    "id": tool_call["id"],
                    "name": tool_call["name"],
                    "args": review_data,
                    "type": "tool_call",
                }
            ],
        )
        return Command(
            goto="run_tool",
            update={"messages": [updated_message]},
        )

    if review_action == "feedback":
        feedback_message = ToolMessage(
            content=review_data,
            tool_call_id=tool_call["id"],
        )
        return Command(
            goto="call_llm",
            update={"messages": [feedback_message]},
        )

    raise ValueError(f"未知审查动作：{review_action}")


review_workflow = StateGraph(ReviewState)
review_workflow.add_node("call_llm", call_llm_for_review)
review_workflow.add_node("human_review", human_review_node)
review_workflow.add_node("run_tool", review_tool_node)
review_workflow.add_edge(START, "call_llm")
review_workflow.add_edge("call_llm", "human_review")
review_workflow.add_edge("run_tool", END)

review_graph = review_workflow.compile(checkpointer=InMemorySaver())


In [ ]:
review_config = {"configurable": {"thread_id": "human-review-demo"}}

first_result = review_graph.invoke(
    {"messages": [HumanMessage(content="帮我查北京天气")]},
    config=review_config,
)

first_result["__interrupt__"]


In [ ]:
# 方式一：批准，直接执行工具。
review_graph.invoke(
    Command(resume={"action": "continue"}),
    config=review_config,
)


In [ ]:
# 方式二：更新工具参数后再执行。
review_config_update = {"configurable": {"thread_id": "human-review-update-demo"}}

review_graph.invoke(
    {"messages": [HumanMessage(content="帮我查北京天气")]},
    config=review_config_update,
)

review_graph.invoke(
    Command(
        resume={
            "action": "update",
            "data": {"query": "北京今天实时天气"},
        }
    ),
    config=review_config_update,
)


## 9. 流式输出

LangGraph 可以边执行边输出中间结果。对于图来说，`.stream()` 会逐步返回节点更新。


In [ ]:
for chunk in simple_graph.stream({"question": "流式输出是什么？"}):
    print(chunk)


## 10. 人机协作、时间旅行、多智能体

这些能力都建立在“图状态 + 检查点”之上：

- 人机协作：在关键节点暂停，等待人工审批或补充信息。
- 时间旅行：从旧检查点恢复执行，观察不同路径。
- 多智能体：把不同 Agent 当作节点或子图，由主管或路由逻辑调度。

入门阶段建议先掌握：`create_agent`、`context_schema`、`StateGraph`、条件边、reducer、checkpointer。


## 11. 常见错误

| 问题 | 原因 | 处理 |
| --- | --- | --- |
| `create_react_agent` 弃用警告 | 还在从 `langgraph.prebuilt` 导入旧入口 | 改用 `from langchain.agents import create_agent` |
| 工具里拿不到 context | 直接调用 `tool.invoke`，没有走 Agent 工具节点 | 用 `agent.invoke(..., context=Context(...))` |
| Agent 不记得前文 | 没有 checkpointer 或 `thread_id` 变化 | 使用 `checkpointer + thread_id` |
| 节点返回值错误 | 节点没有返回状态更新 dict | 返回 `{"字段": 值}` |
| 列表被覆盖 | 没有 reducer | 使用 `Annotated[list[T], add]` 或消息 reducer |


## 12. 参考链接

- LangGraph 快速入门: <https://langgraph.com.cn/agents/agents.1.html>
- LangGraph 图 API 概念: <https://langgraph.com.cn/concepts/low_level.1.html>
- LangGraph 持久化: <https://langgraph.com.cn/concepts/persistence.1.html>
- LangGraph 记忆概念: <https://langgraph.com.cn/concepts/memory.1.html>
- LangGraph 人工参与循环: <https://langgraph.com.cn/concepts/human_in_the_loop.1.html>
- LangGraph 多代理系统: <https://langgraph.com.cn/concepts/multi_agent.1.html>
